<a href="https://colab.research.google.com/github/kaglet/B-Trees-Tool/blob/main/dnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pwd

'/content'

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd ./drive/MyDrive/COS711/classify_galaxies/dev

/content/drive/MyDrive/COS711/classify_galaxies/dev


In [4]:
from sklearn.preprocessing import RobustScaler
import pandas as pd

df = pd.read_csv("../dataset/processed.csv", index_col=0)
df.head()

,modelMag_u,modelMag_z,modelFlux_u,modelFlux_z,petroRad_u,petroRad_g,petroRad_i,petroRad_r,petroRad_z,petroR50_u,...,petroR50_r,petroR50_z,expAB_u,expAB_g,expAB_r,expAB_i,expAB_z,subclass,redshift,redshift_err
0,21.73818,18.23833,2.007378,50.64961,2.969037,4.252946,3.101782,3.461880,3.071923,1.984029,...,1.638081,1.289375,0.099951,0.311864,0.289370,0.270588,0.187182,1,0.067749,0.000015
1,20.66761,18.04122,5.403369,60.73625,2.186902,2.625105,2.678123,2.594866,3.163450,1.069268,...,1.263937,1.318443,0.366549,0.516876,0.517447,0.552297,0.636966,1,0.105118,0.000010
2,23.63531,18.68396,0.713778,33.58972,1.259084,1.644824,1.801951,1.749696,3.059948,0.663606,...,0.987395,1.612933,0.050000,0.417137,0.506950,0.549881,0.370166,1,0.234089,0.000030
3,20.12374,16.72423,8.920645,204.31610,6.625083,4.719598,4.494591,4.777463,4.636094,3.160263,...,2.156205,2.035692,0.310763,0.356827,0.389345,0.388160,0.416660,1,0.110825,0.000030
5,19.47473,16.89580,16.220930,174.45070,4.734951,4.485415,4.086593,4.275559,4.503903,2.455249,...,1.983631,1.863166,0.754158,0.767767,0.759105,0.742471,0.721491,1,0.111458,0.000011


In [5]:
!pip install torch torchvision

In [6]:
import torch

In [7]:
from imblearn.combine import SMOTEENN, SMOTETomek
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

sme = SMOTETomek(random_state=42, sampling_strategy=0.7)
under = RandomUnderSampler(sampling_strategy=1)
pipeline = Pipeline(steps=[('o', sme), ("u", under)])

X = df.drop("subclass", axis=1).to_numpy()
y = df["subclass"].to_numpy().reshape(-1,1)
print((y==0).sum(), (y==1).sum())
print(X.shape, y.shape)

23960 73518
(97478, 20) (97478, 1)


OVERALL TODO:
Too many permutations of parameters to try, takes too long with ANN so try
a linear classifier.

== Preprocess data == (Use pandas cheatsheet if available - DM one too, and site)
~~- Use robust IQR scalar as in paper (handles but do not delete outliers)~~
- Consider using log scale for exponent based (so info not lost through calculations)
~~- Remove redshift errors as in paper~~
~~- Drop obvious columns~~
~~- Create petrosian magnitude column and verify by calculation~~
~~- Drop old petrosian column for flux possibly, seeing how it affects performance~~
~~- Calculate correlation between columns possibly (heatmap may be used)~~
- Shuffle dataset and save for use later
~~- Drop recommended columns (initial) - Justify with domain knowledge~~
~~- Drop columns more aggressively (like that which duplicates what is known)~~
- Perform stratified (proportional) splitting 80/20
- Sample with diff SMOTTEENs or do class weighted regression (see which does better)
- (OPTIONAL) Train with linear classifier or quick decision tree for crude feature selection
-

== Train with 2 ANNs for bigger vs smaller datasets ==
- Get subscription for Colab
- Check how to track when overfitting occurs
- Calculate how big and how deep NNs should be based on intuition (scale about
 proportionally)
- Do NNs with dropout (revise how it works), L2 aggressive regularization, class weights.
- Do these with K-Cross fold training on the dataset (ALL train + test).
- Do multiple K-cross fold to change the weights every time if it does not already.
- Report with confusion matrix, precision/recall/F1.
- Bin by r-mag or redshift showing how faint galaxies affect performance
- From performance consider next steps and review question all steps

- Have architecture ready including ensemble that you can fill and tweak with values
ready essentially is the goal.

In [8]:
import torch.nn as nn
import torch.optim as optim

In [9]:
model_simple = nn.Sequential(
    nn.Linear(21,25),
    nn.ReLU(),
    nn.Linear(25,20),
    nn.ReLU(),
    nn.Linear(20,10),
    nn.ReLU(),
    nn.Linear(10,1),
    nn.Sigmoid()
)
model_simple

Sequential(
  (0): Linear(in_features=21, out_features=25, bias=True)
  (1): ReLU()
  (2): Linear(in_features=25, out_features=20, bias=True)
  (3): ReLU()
  (4): Linear(in_features=20, out_features=10, bias=True)
  (5): ReLU()
  (6): Linear(in_features=10, out_features=1, bias=True)
  (7): Sigmoid()
)

In [166]:
from keras.layers import Dense, LayerNormalization, BatchNormalization
from keras.models import Sequential
from keras.optimizers import Adam
from keras.initializers import GlorotUniform # to use seeds
from keras.activations import leaky_relu # more parametrisation may be excessive for number of samples

In [172]:
def define_model(n_input):
  model = Sequential()
  # 1031 parameters currently
  model.add(Dense(22, input_dim=n_input, activation="tanh", kernel_initializer="glorot_normal"))
  model.add(LayerNormalization())
  model.add(Dense(20, activation="tanh", kernel_initializer="glorot_normal"))
  model.add(LayerNormalization())
  model.add(Dense(15, activation="tanh", kernel_initializer="glorot_normal"))
  model.add(LayerNormalization())
  model.add(Dense(5, activation="tanh", kernel_initializer="glorot_normal"))
  model.add(LayerNormalization())
  model.add(Dense(1, activation="sigmoid", kernel_initializer="glorot_normal"))

  model.compile(loss="binary_crossentropy", optimizer=Adam(learning_rate=0.009, beta_1=0.9, beta_2=0.96))
  model.save_weights("model.weights.h5")
  return model

In [173]:
model = define_model(X.shape[1])
print(model)

<Sequential name=sequential_24, built=True>


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [143]:
# loss_fn = nn.BCELoss()
# optimizer = optim.Adam(model.parameters(), lr=0.005)

In [144]:
import math
import time

In [145]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)
skf.get_n_splits(X, y)

10

In [146]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report

In [171]:
# Can run multiple rounds of cross fold training and report in avg of these
# We get avg of a cross fold training session, then across many sessions
# For now do array of models, one smaller, one larger
# Next do class weighting then SMOTTEEN
scores = []
# Create different splits and on them run training
# I assume this gives list of indices as tuples of splits on shuffled data
# For every k and every other i.e. first time its the first 7 last 1 is test etc.
j = 1
for train_idx, test_idx in skf.split(X,y):
  print(f"Processing Fold set {j}")
  # print(X, y)
  fold_start_time = time.time()
  X_train = X[train_idx, :]
  X_test = X[test_idx, :]
  y_train = y[train_idx]
  y_test = y[test_idx]

  print("PRE-SAMPLING CLASS 0 COUNT = ", (y_train==0).sum(), "\tCLASS 1 COUNT = ", (y_train==1).sum())
  print(X_train.shape, y_train.shape)

  # X_train, y_train = sme.fit_resample(X_train, y_train)
  transformer = RobustScaler().fit(X_train)
  X_train = transformer.transform(X_train)
  X_train = torch.tensor(X_train, dtype=torch.float32)
  y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1,1)
  y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1,1)

  print("SAMPLED CLASS 0 COUNT = ", (y_train==0).sum(), "\tCLASS 1 COUNT = ", (y_train==1).sum())
  print(X_train.shape, y_train.shape)

  n_epochs = 60
  batch_size = 500

  weights = {0:100, 1:70}
  model.fit(X_train, y_train, epochs=n_epochs, class_weight=weights, batch_size=5000, verbose=2)

  with torch.no_grad():
    X_test = transformer.transform(X_test)
    X_test = torch.tensor(X_test, dtype=torch.float32)
    y_pred = model.predict(X_test)

  y_pred = (y_pred > 0.5).astype(int)
  report = classification_report(y_test, y_pred)
  scores.append(report)
  print(f"Fold set {j} took {time.time() - fold_start_time} with scores")
  print(report)

  y_pred = (model.predict(X_train) > 0.5).astype(int)
  report = classification_report(y_train, y_pred)
  print("Report of SEEN data")
  print(report)

  model.load_weights("model.weights.h5")

  j+=1

Processing Fold set 1
PRE-SAMPLING CLASS 0 COUNT =  21564 	CLASS 1 COUNT =  66166
(87730, 20) (87730, 1)
SAMPLED CLASS 0 COUNT =  tensor(21564) 	CLASS 1 COUNT =  tensor(66166)
torch.Size([87730, 20]) torch.Size([87730, 1])
Epoch 1/60
18/18 - 7s - 387ms/step - loss: 41.9225
Epoch 2/60
18/18 - 0s - 5ms/step - loss: 26.7248
Epoch 3/60
18/18 - 0s - 5ms/step - loss: 23.8174
Epoch 4/60
18/18 - 0s - 5ms/step - loss: 22.9830
Epoch 5/60
18/18 - 0s - 5ms/step - loss: 22.5412
Epoch 6/60
18/18 - 0s - 5ms/step - loss: 22.3245
Epoch 7/60
18/18 - 0s - 5ms/step - loss: 22.0888
Epoch 8/60
18/18 - 0s - 5ms/step - loss: 21.8847
Epoch 9/60
18/18 - 0s - 5ms/step - loss: 21.6897
Epoch 10/60
18/18 - 0s - 5ms/step - loss: 21.6137
Epoch 11/60
18/18 - 0s - 5ms/step - loss: 21.2989
Epoch 12/60
18/18 - 0s - 5ms/step - loss: 21.2735
Epoch 13/60
18/18 - 0s - 5ms/step - loss: 20.9931
Epoch 14/60
18/18 - 0s - 5ms/step - loss: 20.9825
Epoch 15/60
18/18 - 0s - 5ms/step - loss: 20.8819
Epoch 16/60
18/18 - 0s - 5ms/step 

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 38 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


18/18 - 0s - 5ms/step - loss: 44.1339
Epoch 2/60
18/18 - 0s - 5ms/step - loss: 26.9215
Epoch 3/60
18/18 - 0s - 5ms/step - loss: 23.8288
Epoch 4/60
18/18 - 0s - 5ms/step - loss: 23.0709
Epoch 5/60
18/18 - 0s - 5ms/step - loss: 22.9392
Epoch 6/60
18/18 - 0s - 5ms/step - loss: 22.5125
Epoch 7/60
18/18 - 0s - 5ms/step - loss: 22.1635
Epoch 8/60
18/18 - 0s - 5ms/step - loss: 22.0424
Epoch 9/60
18/18 - 0s - 5ms/step - loss: 21.8227
Epoch 10/60
18/18 - 0s - 5ms/step - loss: 21.5858
Epoch 11/60
18/18 - 0s - 5ms/step - loss: 21.4394
Epoch 12/60
18/18 - 0s - 5ms/step - loss: 21.3243
Epoch 13/60
18/18 - 0s - 5ms/step - loss: 21.2488
Epoch 14/60
18/18 - 0s - 5ms/step - loss: 20.9825
Epoch 15/60
18/18 - 0s - 5ms/step - loss: 21.1185
Epoch 16/60
18/18 - 0s - 5ms/step - loss: 20.9341
Epoch 17/60
18/18 - 0s - 5ms/step - loss: 20.7544
Epoch 18/60
18/18 - 0s - 5ms/step - loss: 20.7477
Epoch 19/60
18/18 - 0s - 5ms/step - loss: 20.6858
Epoch 20/60
18/18 - 0s - 5ms/step - loss: 20.6103
Epoch 21/60
18/18 - 

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 38 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


18/18 - 0s - 5ms/step - loss: 44.8727
Epoch 2/60
18/18 - 0s - 5ms/step - loss: 27.4297
Epoch 3/60
18/18 - 0s - 5ms/step - loss: 23.9166
Epoch 4/60
18/18 - 0s - 6ms/step - loss: 22.9803
Epoch 5/60
18/18 - 0s - 6ms/step - loss: 22.5876
Epoch 6/60
18/18 - 0s - 6ms/step - loss: 22.3781
Epoch 7/60
18/18 - 0s - 5ms/step - loss: 22.1418
Epoch 8/60
18/18 - 0s - 5ms/step - loss: 21.8560
Epoch 9/60
18/18 - 0s - 5ms/step - loss: 21.6849
Epoch 10/60
18/18 - 0s - 5ms/step - loss: 21.4959
Epoch 11/60
18/18 - 0s - 6ms/step - loss: 21.2923
Epoch 12/60
18/18 - 0s - 6ms/step - loss: 21.2324
Epoch 13/60
18/18 - 0s - 6ms/step - loss: 21.1141
Epoch 14/60
18/18 - 0s - 6ms/step - loss: 21.0672
Epoch 15/60
18/18 - 0s - 5ms/step - loss: 20.9417
Epoch 16/60
18/18 - 0s - 6ms/step - loss: 20.8284
Epoch 17/60
18/18 - 0s - 5ms/step - loss: 20.8527
Epoch 18/60
18/18 - 0s - 5ms/step - loss: 20.6544
Epoch 19/60
18/18 - 0s - 5ms/step - loss: 20.6158
Epoch 20/60
18/18 - 0s - 5ms/step - loss: 20.5255
Epoch 21/60
18/18 - 

...

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 38 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


18/18 - 0s - 5ms/step - loss: 43.9818
Epoch 2/60
18/18 - 0s - 5ms/step - loss: 26.9439
Epoch 3/60
18/18 - 0s - 5ms/step - loss: 23.8682
Epoch 4/60
18/18 - 0s - 5ms/step - loss: 23.0846
Epoch 5/60
18/18 - 0s - 5ms/step - loss: 22.7629
Epoch 6/60
18/18 - 0s - 5ms/step - loss: 22.3822
Epoch 7/60
18/18 - 0s - 5ms/step - loss: 22.0656
Epoch 8/60
18/18 - 0s - 5ms/step - loss: 21.9099
Epoch 9/60
18/18 - 0s - 5ms/step - loss: 21.7437
Epoch 10/60
18/18 - 0s - 5ms/step - loss: 21.5513
Epoch 11/60
18/18 - 0s - 5ms/step - loss: 21.3575
Epoch 12/60
18/18 - 0s - 5ms/step - loss: 21.1974
Epoch 13/60
18/18 - 0s - 5ms/step - loss: 20.9741
Epoch 14/60
18/18 - 0s - 5ms/step - loss: 20.9736
Epoch 15/60
18/18 - 0s - 5ms/step - loss: 20.8644
Epoch 16/60
18/18 - 0s - 5ms/step - loss: 20.6789
Epoch 17/60
18/18 - 0s - 5ms/step - loss: 20.6771
Epoch 18/60
18/18 - 0s - 5ms/step - loss: 20.6661
Epoch 19/60
18/18 - 0s - 5ms/step - loss: 20.6415
Epoch 20/60
18/18 - 0s - 5ms/step - loss: 20.5189
Epoch 21/60
18/18 - 

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 38 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


18/18 - 0s - 5ms/step - loss: 44.3967
Epoch 2/60
18/18 - 0s - 5ms/step - loss: 26.8629
Epoch 3/60
18/18 - 0s - 5ms/step - loss: 23.8603
Epoch 4/60
18/18 - 0s - 6ms/step - loss: 23.0313
Epoch 5/60
18/18 - 0s - 6ms/step - loss: 22.6808
Epoch 6/60
18/18 - 0s - 5ms/step - loss: 22.4187
Epoch 7/60
18/18 - 0s - 6ms/step - loss: 22.3293
Epoch 8/60
18/18 - 0s - 6ms/step - loss: 22.1053
Epoch 9/60
18/18 - 0s - 6ms/step - loss: 21.8381
Epoch 10/60
18/18 - 0s - 5ms/step - loss: 21.8732
Epoch 11/60
18/18 - 0s - 6ms/step - loss: 21.6502
Epoch 12/60
18/18 - 0s - 6ms/step - loss: 21.4125
Epoch 13/60
18/18 - 0s - 6ms/step - loss: 21.1542
Epoch 14/60
18/18 - 0s - 6ms/step - loss: 21.1518
Epoch 15/60
18/18 - 0s - 6ms/step - loss: 20.9450
Epoch 16/60
18/18 - 0s - 6ms/step - loss: 20.8303
Epoch 17/60
18/18 - 0s - 6ms/step - loss: 20.7135
Epoch 18/60
18/18 - 0s - 5ms/step - loss: 20.6956
Epoch 19/60
18/18 - 0s - 5ms/step - loss: 20.5456
Epoch 20/60
18/18 - 0s - 5ms/step - loss: 20.6099
Epoch 21/60
18/18 - 

KeyboardInterrupt: 

In [ ]:
scores

This is for dataset size of 100000/100. It's eerily good at classifying unseen data lol.

[tensor(0.8523),
 tensor(0.8469),
 tensor(0.8532),
 tensor(0.8503),
 tensor(0.8529),
 tensor(0.8520),
 tensor(0.8544),
 tensor(0.8510)]